<a href="https://colab.research.google.com/github/Liping-LZ/BDAO_DSDO/blob/main/Week%201/BDAO_Block1_Workshop_FakeStore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BDAO Block 1 — API Workshop: FakeStore API
### Exploring an Ecommerce API

**FakeStore API:** https://fakestoreapi.com/  
A free, open ecommerce API — no authentication needed. Simulates a real online store with products, orders, carts and users.

---
## How this workshop works

This notebook has **three stages**:

| Stage | What you do |
|---|---|
| **Explore** | Call the API, understand what data is available |
| **Analyse** | Extract insights from the data |
| **Connect** | Combine data from two endpoints to answer a business question |

Each task has:
- A **business question** to answer
- A **YOUR CODE** block to complete
- A **HINT** cell you can uncomment if you get stuck

The goal is not to finish everything — it is to understand what you are doing and why.

---
> **Remember from the demo:**  
> Every API call is: `response = requests.get(url, params=params)`  
> The response is JSON: `data = response.json()`  
> JSON is a Python dictionary: navigate it with `data['key']`

## Setup — run this first

In [ ]:
import requests
import json
import pandas as pd

pd.set_option('display.max_columns', 12)
pd.set_option('display.max_colwidth', 45)
pd.set_option('display.float_format', '{:.2f}'.format)

# Base URL — all FakeStore endpoints start here
BASE = 'https://fakestoreapi.com'

print('Ready. Base URL:', BASE)
print()
print('Documentation: https://fakestoreapi.com/docs')

---
## Stage 1 — EXPLORE: Understand the API

Before writing any analysis code, you need to understand what the API returns.  
This stage is about reading the data, not processing it.

### Task 1.1 — What does the products endpoint return?

Call the products endpoint and look at the raw response for one product.  
**Before running:** what fields do you expect a product to have?

In [ ]:
# Call the products endpoint
response = requests.get(f'{BASE}/products')
products = response.json()

print(f'Status code: {response.status_code}')
print(f'Number of products returned: {len(products)}')
print()
print('First product (raw JSON):')
print(json.dumps(products[0], indent=2))

In [ ]:
# What fields does each product have?
products[0].keys()

In [ ]:
# What fields does each product have?
print('Fields in each product:')
for key, value in products[0].items():
    print(f'  {key:<15} → {type(value).__name__}')

### Task 1.2 — What categories exist?

Use the `/products/categories` endpoint to get all available categories.  
**Business question:** how many product categories does this store have?

In [ ]:
# Your turn: Call /products/categories and print the result

# YOUR CODE HERE
categories = None  # replace None with your API call

print('Available categories:')
# YOUR CODE: print each category


### Task 1.3 — Get products from a specific category

The endpoint `/products/category/{category_name}` returns products in one category.  
**Business question:** what products does this store sell in the electronics category?

In [ ]:
# YOUR CODE: Get all products in the 'electronics' category
# Endpoint: /products/category/electronics

# YOUR CODE HERE
electronics = None # replace None with your API call

# Print each product name and price
print('Electronics products:')
# YOUR CODE: loop through electronics and print name + price


---
## Stage 2 — ANALYSE: Extract business insights

Now you have explored the data, use it to answer real business questions.  
This is what a data analyst does with API data every day.

### Task 2.1 — Build a product catalogue DataFrame

**Business question:** give the buying team a clean summary of all products with their prices and ratings.

In [ ]:
# YOUR CODE: Build a DataFrame from all products
# Extract: id, title, price, category, rating.rate, rating.count
# Remember: rating is nested — access it with p['rating']['rate']

records = []
for p in products:
    records.append({
        'id':           p['id'],
        'title':        p['title'][:45],
        'price':        p['price'],
        'category':     p['category'],
        # YOUR CODE: add rating_score and rating_count
    })

df_products = pd.DataFrame(records)
print(f'Product catalogue: {len(df_products)} products')
df_products

### Task 2.2 — Which category has the highest average price?

**Business question:** the pricing team wants to know which category commands the highest prices on average.

In [ ]:
# YOUR CODE: Group by category, calculate average price, sort highest first
# Use df_products.groupby()

category_prices = None # YOUR CODE HERE - replace None with your code

print('Average price by category:')
print(category_prices)

### Task 2.3 — Find the best value products

**Business question:** the marketing team wants to promote products with high ratings but low prices — best value items.  
Find products with a rating above 4.0 and a price below the median price.

In [ ]:
# YOUR CODE: Filter for high rating AND low price
# Step 1: calculate median price
# Step 2: filter where rating_score > 4.0 AND price < median
# Step 3: sort by rating_score descending

# YOUR CODE HERE
best_value = df_products  # complete the code to filter the data based on the given conditions

print(f'Best value products (high rating, below median price):')
best_value[['title','category','price','rating_score','rating_count']]

---
## Stage 3 — CONNECT: Combine two API endpoints

The real power of APIs is combining data from multiple endpoints.  
A single API call gives you one perspective — combining them gives you business insight.

We will now call the **carts endpoint** (orders) and join it with the products data.

### Task 3.1 — Understand the carts endpoint

Call `/carts` and look at the structure.  
**Before running:** what do you think a cart (order) contains?

In [ ]:
# Call the carts endpoint
carts = requests.get(f'{BASE}/carts').json()

print(f'Number of carts (orders): {len(carts)}')
print()
print('First cart (raw):')
print(json.dumps(carts[0], indent=2))

In [ ]:
# Build an order lines table
# Each cart can have multiple products — we unpack them into one row per product

order_lines = []
for cart in carts:
    for item in cart['products']:
        order_lines.append({
            'cart_id':    cart['id'],
            'user_id':    cart['userId'],
            'date':       cart['date'],
            'product_id': item['productId'],
            'quantity':   item['quantity']
        })

df_orders = pd.DataFrame(order_lines)
print(f'Order lines: {len(df_orders)} (one row per product per cart)')
df_orders.head(6)

### Task 3.2 — Join orders to products

**Business question:** the revenue team needs to know which categories generate the most sales.  
Join the orders table to the products table so each order line shows the product name, category and price.

In [ ]:
# YOUR CODE: Join df_orders to df_products
# Use pd.merge() — join on product_id (orders) and id (products)


df_joined = pd.merge()

# Then calculate revenue = price * quantity
# YOUR CODE: add a revenue column
df_joined['revenue'] = None  # replace with price * quantity

print(f'Joined order lines: {len(df_joined)}')
df_joined[['cart_id','product_id','title','category','quantity','price','revenue']].head(8)

### Task 3.3 — Revenue by category

**Business question:** which product category generates the most revenue across all orders?

In [ ]:
# YOUR CODE: Group by category, sum revenue, sort descending

revenue_by_category = (
    df_joined
    # YOUR CODE HERE
)

print('Revenue by category:')
revenue_by_category

---
## Reflection — think about these before the debrief

Write your answers here (double-click to edit):

**Q1.** This API has 20 products and 7 carts. A real ecommerce platform has millions.  
What would break in this notebook if the API returned 1 million products?

*Your answer:*

---

**Q2.** We called three separate endpoints: `/products`, `/products/categories`, `/carts`.  
In a real business pipeline, these calls might run every hour automatically.  
What infrastructure would you need to make that happen?

*Your answer:*

---

**Q3.** Look at your revenue by category result.  
What is your one-sentence business insight?  
Who in the company would you present this to?

*Your answer:*

---

**Q4.** The Spotify API required authentication — FakeStore did not.  
Why might a real ecommerce company require API authentication even for public product data?

*Your answer:*

---
## Extension — if you finish early

**Extension A:** Call the `/users` endpoint.  
Which city has the most customers? What is the most common email domain?

In [ ]:
# Extension A: Explore the users endpoint
users = requests.get(f'{BASE}/users').json()
print(f'Users: {len(users)}')
print(json.dumps(users[0], indent=2))

# YOUR CODE: find most common city and email domain

**Extension B:** Who is the highest-spending customer?  
Join the carts data to the users data and calculate total spend per customer.

In [ ]:
# Extension B: Customer spend analysis
# Step 1: build a user DataFrame from the users endpoint
# Step 2: join df_joined (which has user_id and revenue) to users
# Step 3: sum revenue per customer

# YOUR CODE HERE
print('Top customers by spend:')

**Extension C:** The FakeStore API also has a sort parameter.  
Try: `GET /products?sort=desc` — what changes?  
And a limit parameter: `GET /products?limit=5` — what does it return?

In [ ]:
# Extension C: Query parameters
# YOUR CODE: call /products with sort=desc and limit=5

sorted_products = requests.get(f'{BASE}/products', params={'sort': 'desc', 'limit': 5}).json()
print(f'Returned {len(sorted_products)} products:')
for p in sorted_products:
    print(f"  id={p['id']}  £{p['price']:.2f}  {p['title'][:40]}")